# Demographics and Building Damage: Who Gets Hit Hardest?

## Introduction

In Lessons 2 and 3, we predicted building damage using only **structural features** — foundation type, roof materials, age, and height. But earthquakes affect **people**, not just buildings. Today we ask the more difficult question: **Who gets hit hardest?**

**The key question for this lesson:** do demographic factors like caste, income, and household size affect earthquake damage outcomes? Or is the damage explained entirely by building structure? This distinction matters enormously for policy:

- If damage is purely structural → repair programs should target old buildings with weak foundations
- If damage correlates with demographics → equity programs are needed alongside structural repairs; certain communities may be systematically underserved by pre-earthquake building standards

The 2015 Nepal Gorkha earthquake exposed deep inequalities. Rural communities, lower-income households, and historically marginalized caste groups suffered disproportionately. Our data science task is to *quantify* those inequalities — not to editorialize, but to make the patterns visible so policy makers can act.

By the end of this lesson, you will be able to:

1. Define the **ML framing** problem — specifying targets, binary classification, and handling class imbalance
2. Identify and prevent **target leakage** — the most common silent failure in data science
3. Query relational databases with **SQL JOINs, GROUP BY, HAVING, and table aliases**
4. Perform **exploratory data analysis (EDA)** on joined demographic and building data
5. Handle **high-cardinality categorical features** (like caste with 100+ distinct values)
6. Build a classification model with **both structural and demographic features**
7. Analyze damage patterns by **municipality** to understand spatial heterogeneity
8. Conduct a **caste-based equity analysis** — who lives in damaged buildings?


## Part 1: ML Framing — Defining the Problem

Before writing any code, we need to clearly define what we're predicting and why.

### The Prediction Target

A **target** (or **label**) is what we want to predict. In this project:

```
Target Variable: severe_damage
  ↓
  Values: 0 or 1 (binary classification)
  ↓
  0 = Grade 1, 2, or 3 (intact, minor, moderate damage — building can be occupied)
  1 = Grade 4 or 5 (heavily damaged or collapsed — building must be vacated)
```

The target comes from the `building_damage` table's `damage_grade` column, which records post-earthquake structural assessments.

### Why Binary Classification?

The raw data has **5 damage grades** (a quasi-continuous scale). We convert to binary (two classes) for practical reasons:

| Reason | Explanation |
|---|---|
| **Clear policy boundary** | The rebuilding/evacuation decision is binary: safe to occupy vs. not safe |
| **Emergency response** | First responders need "Can people enter this building?" — not a 1-5 score |
| **Class balance** | 5-class problem would be severely imbalanced; binary is more tractable |
| **Simplicity** | Binary predictions are easier to explain to non-technical stakeholders |
| **Real-world use** | Disaster response agencies issue binary occupancy certificates |

### Converting Damage Grades to Binary

Original `damage_grade` values:

```
Grade 1: Negligible to slight damage   →  severe_damage = 0
Grade 2: Moderate damage               →  severe_damage = 0
Grade 3: Moderate to heavy damage      →  severe_damage = 0
Grade 4: Heavy damage                  →  severe_damage = 1
Grade 5: Complete/near-complete collapse → severe_damage = 1
```

**The conversion logic reflects policy:** the threshold between Grade 3 and Grade 4 is where buildings become uninhabitable. Grade 3 buildings may need repairs but occupants can stay. Grade 4+ buildings must be vacated and rebuilt.

> 💡 **Every binary classification choice is a policy decision.** Where we draw the line between 0 and 1 determines which buildings get flagged for intervention. A more conservative threshold (Grade 3+ = severe) would flag more buildings but at higher cost. A looser threshold (Grade 5 only = severe) would miss buildings in urgent need. The threshold should be chosen to match real-world consequences.


### Class Imbalance

**Class imbalance** occurs when one class substantially outnumbers another.

```
Gorkha District (70,836 buildings):
  Severely damaged (Grade 4-5): 45,493 buildings  → 64%  ← Majority class
  Not severely damaged (Grade 1-3): 25,343 buildings → 36% ← Minority class

  Imbalance ratio: 1.8:1 — relatively mild
```

> ⚠️ **The accuracy trap:** A "model" that always predicts "severe" (never looking at features) achieves **64% accuracy** on Gorkha data — because 64% of buildings ARE severe. This is the **majority-class baseline**. Any real model must beat 64% to be useful.

**Why class imbalance matters:**

- Accuracy alone is misleading — a model can look good while being useless for the minority class
- **Precision**: of buildings we predict as "not severe," how many truly are safe?
- **Recall**: of truly severely damaged buildings, what fraction do we correctly flag?
- **F1-score**: harmonic mean of precision and recall — balances both concerns

**The disaster-response asymmetry:** false negatives (predicting "safe" when the building is actually severely damaged) are far more costly than false positives (predicting "severe" when the building is actually fine). In emergency response, we accept more false alarms to ensure we don't miss a collapsed building where people might be trapped.

**Mitigation strategies:**
1. Use `class_weight='balanced'` in sklearn classifiers
2. Lower the decision threshold (e.g., 0.4 instead of 0.5) to increase recall
3. Use stratified splits to ensure train and test sets have the same class proportions
4. Report recall and F1-score alongside accuracy


### Target Leakage: The Silent Killer

**Target leakage** (also called **data leakage**) occurs when information about the target variable leaks into the features — specifically, information that would *not* exist at prediction time.

> ⚠️ **Why it's called "the silent killer":** leaky models look excellent during development (training accuracy 99%!) but fail catastrophically in production (test accuracy 65%). There is no error message, no warning. The model trains successfully, evaluates well, and then fails when it matters most.

#### Leakage Scenario 1: Post-Earthquake Features

Imagine including these columns in our feature matrix:

```
post_eq_floor_count:    floor count measured AFTER the earthquake
post_eq_roof_type:      roof type assessed AFTER earthquake
damage_description:     text assessment created AFTER assessment visit
repair_cost:            estimated only AFTER damage is known
```

**Why this is leakage:** at prediction time (before or during an earthquake), we do not have post-event measurements. The model would learn to use "post_eq_floor_count went from 3 to 1" as a signal — which is essentially the same as knowing the building collapsed.

**The result:**
```
With leakage:
  Training accuracy: 99%   ← looks amazing!
  Test accuracy:     68%   ← real world crashes

Without leakage:
  Training accuracy: 74%
  Test accuracy:     71%   ← consistent, trustworthy
```

> 📌 **The governing rule:** every feature must represent information that exists BEFORE the event we are trying to predict. For earthquake damage prediction: if the feature was measured or generated after the earthquake, it cannot be used.


#### Leakage Scenario 2: Direct Target Proxy

Some features directly restate the target:

```
damage_description_text = "Building completely collapsed"
```

This is almost perfectly correlated with `severe_damage = 1`. The model would learn to read the description and output the grade — achieving near-perfect accuracy, but providing zero new information. The model would have no predictive power for buildings where no assessment has happened yet.

#### Leakage Scenario 3: Data Pipeline Leakage (Encoder on Full Dataset)

This is the most subtle — and most common — form of leakage:

```python
# WRONG: encoder sees full dataset including test set
encoder = OrdinalEncoder()
X_encoded = encoder.fit_transform(X)          # ← full data leak!
X_train, X_test = train_test_split(X_encoded)

# CORRECT: encoder sees only training data
X_train, X_test = train_test_split(X)
encoder = OrdinalEncoder()
X_train_enc = encoder.fit_transform(X_train)  # fit on train only
X_test_enc  = encoder.transform(X_test)        # transform without fitting
```

When the encoder fits on the full dataset (including test), it uses test-set statistics to inform the encoding — a subtle form of information leakage from test to train.

> ⚠️ **This is why we use `Pipeline`.** A `sklearn.Pipeline` automatically applies `fit_transform` only during `fit()`, and only `transform` during `predict()`. It enforces the correct train-only fitting discipline without manual code management.

#### Prevention Checklist

For every feature, ask:

1. ✅ **Temporal**: "Does this information exist BEFORE the earthquake event?"
2. ✅ **Causal**: "Was this feature caused BY the damage, not the other way around?"
3. ✅ **Pipeline**: "Is the encoding fit only on training data?"

**Features we DROP due to leakage in this lesson:**
```python
cols_to_drop = [
    'post_eq_floor_count',  # Measured after earthquake
    'post_eq_roof_type',    # Assessed after earthquake
    'damage_grade',         # This IS the target (raw form)
    'building_id',          # Arbitrary ID, not a feature
]
```


## Part 2: Relational Databases and Schema Design

Our earthquake data lives in a **relational database**. Understanding the schema prevents query errors and enables efficient joins.

### The Nepal Earthquake Database Schema

The database has **4 interconnected tables**:

```
┌────────────────────────────────────────────────────────────────────┐
│                    RELATIONAL DATABASE SCHEMA                       │
└────────────────────────────────────────────────────────────────────┘

┌─────────────────────────┐          ┌─────────────────────────┐
│  building_structure      │          │  household_demographics  │
├─────────────────────────┤          ├─────────────────────────┤
│ ⭐ building_id (PK)     │          │ ⭐ household_id (PK)    │
│   age_building          │          │   caste_household        │
│   foundation_type       │          │   income_household       │
│   ground_floor_type     │          │   members_household      │
│   roof_type             │          │   rooms                  │
│   height_ft_pre_eq      │          │                         │
│   plinth_area_sq_ft     │          │                         │
└─────────────────────────┘          └─────────────────────────┘
          △                                       △
          │                                       │
          │ building_id                           │ household_id
          └───────────────┬───────────────────────┘
                          │
                 ┌────────▼──────────┐
                 │      id_map       │  ← Bridge table
                 ├───────────────────┤
                 │ ⭐ household_id (FK)
                 │ ⭐ building_id (FK)
                 │   district_id
                 │   vdcmun_id
                 └───────────────────┘
                          △
                          │ building_id
                 ┌────────┴──────────┐
                 │  building_damage  │
                 ├───────────────────┤
                 │ ⭐ building_id (PK)
                 │   damage_grade    │
                 │   damage_count    │
                 └───────────────────┘

Legend:
⭐ = Primary Key (unique identifier for this table)
FK = Foreign Key (reference to another table)
```

### Why a Bridge Table?

`building_structure` and `household_demographics` have different primary keys — they cannot be directly joined. The `id_map` table provides the connection:

```
building_structure:       household_demographics:
  building_id = 164817       household_id = 1001
  age_building = 15          caste = "Gurung"
  foundation = "RC"          income = 50000

id_map:
  household_id = 1001  ─── links to ───  building_id = 164817
  district_id = 4                         vdcmun_id = 12
```

One building can house **multiple households** (1-to-many relationship). The bridge table stores location data (`district_id`, `vdcmun_id`) for each building-household pair.

> 📌 **Relational design prevents data duplication.** Building structural information is stored once in `building_structure`, not repeated for each household in that building. If a building's foundation type is re-classified after inspection, we update one row — not hundreds.


## Part 3: SQL JOINs, HAVING, and Table Aliases

### INNER JOIN: Matching Records from Multiple Tables

An `INNER JOIN` returns only rows where both tables have a matching key — unmatched rows are dropped.

```sql
SELECT h.household_id, h.caste_household, s.foundation_type, d.damage_grade
FROM household_demographics h
  INNER JOIN id_map i ON h.household_id = i.household_id
  INNER JOIN building_structure s ON i.building_id = s.building_id
  INNER JOIN building_damage d ON i.building_id = d.building_id
WHERE i.district_id = 4
LIMIT 5;
```

**Reading the chain:**
```
Start: household_demographics (alias h)
    ↓ Join on household_id → id_map (alias i)
    ↓ Join on building_id → building_structure (alias s)
    ↓ Join on building_id → building_damage (alias d)
    Filter: district_id = 4 (Gorkha district only)
```

**Why INNER JOIN drops rows:** if a household has no matching building in `id_map`, or a building has no matching damage record in `building_damage`, that household is excluded from results. This is correct behavior — incomplete records would introduce missing values into our feature matrix.

### Table Aliases: Shorthand References

Aliases (`h`, `i`, `s`, `d`) are shorthand for table names, defined immediately after the table name:

```sql
FROM household_demographics h   ← h is an alias for household_demographics
```

Benefits:
- `h.household_id` is shorter than `household_demographics.household_id`
- Essential when the same table appears multiple times in a query
- Prevents ambiguity when multiple tables have columns with the same name

### GROUP BY and HAVING: Aggregation with Filters

**Scenario:** "Which castes have the most households in Gorkha?"

```sql
SELECT caste_household, COUNT(*) AS household_count
FROM household_demographics
GROUP BY caste_household
HAVING COUNT(*) > 100          -- Only castes with 100+ households
ORDER BY household_count DESC;
```

**The critical difference: WHERE vs HAVING:**

```
WHERE filters rows BEFORE grouping:
  SELECT caste, COUNT(*) FROM households
  WHERE income > 50000          -- Keep only wealthy households first
  GROUP BY caste                -- Then group

HAVING filters groups AFTER grouping:
  SELECT caste, COUNT(*) FROM households
  GROUP BY caste                -- Group first
  HAVING COUNT(*) > 1000        -- Then keep only large castes
```

> 📌 **Rule:** `WHERE` operates on individual rows; `HAVING` operates on aggregate groups. Use `HAVING` whenever your filter involves an aggregate function (`COUNT`, `SUM`, `AVG`, etc.).


## Part 4: Exploratory Data Analysis and the Wrangling Workflow

### EDA: Understanding the Data Before Modeling

**Exploratory Data Analysis (EDA)** is the process of investigating the data for patterns, anomalies, and relationships before fitting any model. It prevents surprises.

**Key EDA questions for L4:**

1. **Distributions:** what does the caste distribution look like? Are most households concentrated in a few castes?
2. **Missing data:** are any columns missing values? How many?
3. **Target relationship:** do certain castes have higher severe damage rates? Is this due to caste itself, or because certain castes live in older/weaker buildings?
4. **Cardinality:** how many unique castes are there? (Expect 100+)
5. **Spatial patterns:** do damage rates vary by municipality (`vdcmun_id`)?

**EDA code patterns you'll use:**

```python
df.info()                          # dtypes, non-null counts
df.isnull().sum()                  # missing values per column
df['severe_damage'].value_counts() # class distribution
df['caste_household'].nunique()    # unique caste count
df.groupby('caste_household')['severe_damage'].mean()  # damage rate by caste
```

### The Data Wrangling Pipeline

**Wrangling** transforms raw SQL query output into a clean, model-ready DataFrame. Every step has a reason:

```
Raw SQL output
  ↓ Drop leaky columns (post_eq_*, damage_grade)
  ↓ Drop rows with missing values
  ↓ Fix data types (object → category or int)
  ↓ Handle high-cardinality caste column (top 10 + "Other")
  ↓ Create binary target (damage_grade → severe_damage)
  ↓ Split train/test (BEFORE encoding)
  ↓ Encode categoricals (ONLY on training data)
  ↓ Model training
```

**Why split BEFORE encoding?**

```python
# WRONG (pipeline leakage):
encoder.fit(X)                  # Sees test statistics
X_train, X_test = split(...)

# CORRECT (no leakage):
X_train, X_test = split(...)
encoder.fit(X_train)            # Only training statistics
encoder.transform(X_test)       # Apply without re-fitting
```

### High-Cardinality Categoricals: The Caste Problem

The `caste_household` column has **100+ distinct caste names** in Nepal. One-hot encoding all of them would create 100+ binary columns — many with very few observations.

**Problems with many rare categories:**
- Rare categories get very few training examples → model can't learn reliable patterns
- Many extra columns → overfitting risk, slower training
- Test data may have categories unseen during training → encoder errors

**Solution: Keep top-N, group the rest:**

```python
top_10_castes = df['caste_household'].value_counts().head(10).index
df['caste_household'] = df['caste_household'].apply(
    lambda c: c if c in top_10_castes else 'Other'
)
```

This reduces 100+ categories to 11 (top 10 + "Other"), making the encoding tractable without losing the most important variation.


## Part 5: The Equity Question — Caste, Demographics, and Damage

This lesson goes beyond accuracy metrics to ask a policy question: **does caste predict earthquake damage, and if so, why?**

### The Nepal Caste System and Structural Inequality

Nepal's caste system has centuries-old roots. Historically, lower-caste communities (Dalits, some Janajati groups) were excluded from land ownership, credit access, and government services. This translated into:

- **Older housing stock** — wealthier groups rebuilt with stronger materials (RC); lower-caste families maintained older mud-mortar or stone construction
- **Rural vs urban location** — lower-caste communities more often in seismically exposed areas outside city cores
- **Less access to earthquake-safe construction** — fewer resources for retrofitting or rebuilding before 2015

> 📌 **The key analytical challenge:** if Dalit households have higher severe damage rates, is this because being Dalit *causes* more damage? No — it is because caste correlates with building quality, location, and access to retrofitting resources. The relationship is **mediated** by structural and economic factors, not direct.

### Three Possible Findings and Their Policy Implications

| Finding | Interpretation | Policy Implication |
|---------|---------------|-------------------|
| **Caste predicts damage even after controlling for structure** | Demographic factors have independent effect (e.g., through maintenance, occupancy patterns) | Equity programs needed alongside structural repairs |
| **Caste predicts damage but NOT after controlling for structure** | Effect is fully mediated by building quality | Target structural repair programs by building type, reaching caste communities indirectly |
| **No caste-damage relationship** | Damage is purely random or driven by geography | Focus on geographic targeting |

> 📌 **Our analysis will quantify which of these holds for Gorkha.** The answer matters for aid prioritization: does Nepal's reconstruction budget prioritize building-type targeting or community-targeting?

### Municipality-Level Analysis

Damage also varies by **municipality** (`vdcmun_id`). Some municipalities had near-total destruction; others were relatively spared. This spatial heterogeneity matters because:

- Certain castes are concentrated in specific municipalities
- A caste-damage correlation might be a municipality-damage correlation in disguise
- Policy must target the right geographic unit


## Part 6: Benchmarking Rigor

A rigorous machine learning workflow includes clear benchmarks at each stage.

### The Benchmark Ladder

```
Step 1: Majority-class baseline
  → Predict "severe" for every building: 64% accuracy
  → This is the floor — any model must beat this

Step 2: Logistic Regression (structure features only)
  → ~71% accuracy (from Lesson 2)
  → Baseline for adding demographic features

Step 3: Logistic Regression (structure + demographics)
  → Does adding caste/income improve accuracy?
  → If not, demographics add no predictive signal

Step 4: Decision Tree (structure + demographics)
  → Does non-linearity help with demographic data?
```

### Evaluation Standards

| Metric | What it Measures | When to Use |
|--------|-----------------|-------------|
| **Accuracy** | Fraction correct overall | Only when classes are balanced |
| **Precision** | Of "not severe" predictions, how many right? | When false alarms are costly |
| **Recall** | Of truly severe buildings, fraction caught | When misses are costly (our case!) |
| **F1-score** | Harmonic mean of precision and recall | Default for imbalanced classification |
| **Test set performance** | Generalization to unseen buildings | Always — the only metric that matters for deployment |

### Red Flags in Model Evaluation

```
🚩 Training accuracy 99%, test accuracy 65%   → Severe overfitting
🚩 Reporting only accuracy with 64/36 imbalance → Misleading (baseline gets 64%!)
🚩 Evaluating on training data only             → Optimistically biased
🚩 Choosing model architecture after seeing test set → Implicit leakage
```

> 💡 **Good models are built on good processes.** A model evaluated correctly at 71% accuracy is more valuable than a model evaluated incorrectly at 85% accuracy. The latter will fail in deployment.


**Code 4.4.0.1**: Import Libraries

This lesson uses:
- **`pandas`**, **`numpy`**: data manipulation
- **`matplotlib.pyplot`**: plotting
- **`duckdb`**: connecting to the SQLite database
- **`sklearn.linear_model.LogisticRegression`**: classification model
- **`sklearn.model_selection.train_test_split`**: splitting data
- **`sklearn.pipeline.Pipeline`**: chaining encoder + classifier
- **`sklearn.metrics.accuracy_score`**: evaluation
- **`category_encoders.OneHotEncoder`**: encoding with `use_cat_names=True`


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import duckdb
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from category_encoders import OneHotEncoder
from sklearn.pipeline import Pipeline

# Set display options
pd.set_option('display.max_columns', None)

---

## Part 7: Exploring the Demographics Table

We start by examining the `household_demographics` table directly — before any joins. This gives us a feel for the raw demographic data.

> 📌 **Why explore raw tables first?** Before joining, verify that the table looks as expected — correct column names, sensible values, no obvious data quality issues. A bug in a join query is much harder to diagnose than a bug in a single-table query.

**Code Task 4.4.1.1**: Connect to the SQLite database using `duckdb.connect('./nepal.sqlite')`. Query the `household_demographics` table and select the first 5 rows. Store the result as `df_demo`.


In [ ]:
# Connect to database
conn = duckdb.connect('./nepal.sqlite')

# Query household_demographics table
df_demo = conn.execute("""
    SELECT *
    FROM household_demographics
    LIMIT 5
""").df()

conn.close()

print(f"Shape: {df_demo.shape}")
print("\nColumns:", df_demo.columns.tolist())
print("\nFirst 5 rows:")
print(df_demo)

Now let's count the total number of rows in the `household_demographics` table to understand the scale of the data.

> 💡 **Scale check:** we expect ~70,000 households (one per household in Gorkha district). This is different from buildings — one building can house multiple households.

**Code Task 4.4.1.2**: Write a SQL query to count how many observations are in the `household_demographics` table. Store the count as `count_households`.


In [ ]:
conn = duckdb.connect('./nepal.sqlite')

count_df = conn.execute("""
    SELECT COUNT(*) as count
    FROM household_demographics
""").df()

conn.close()

demo_count = count_df.loc[0, 'count']
print(f"Number of households: {demo_count}")

✅ **You may now attempt Multiple Choice Question 4.4.1.1**

> 📊 **Interpreting the count:** the household count should be larger than the building count (~70,836). Why? One building can house multiple households (a family compound with separate households). The `id_map` table bridges the one-to-many relationship.

---

## Part 8: Joining Tables — Getting the Full Picture

To build a model combining structural and demographic features, we need to join all four tables:

- `household_demographics` — occupant information (caste, income, household size)
- `building_structure` — structural features (foundation, roof, age, height)
- `building_damage` — damage grades
- `id_map` — the bridge table linking all three

**The JOIN chain:**
```
household_demographics
  → (via household_id) → id_map
  → (via building_id)  → building_structure
  → (via building_id)  → building_damage
```

**Important:** we use `INNER JOIN`, which drops any households or buildings that don't appear in all four tables. This ensures a complete record for every row.

**Code Task 4.4.2.1**: Write a SQL query that joins all four tables. Select all columns from `household_demographics` (prefix `h.*`) and `building_structure` (prefix `s.*`), plus `vdcmun_id` and `district_id` from `id_map`, plus `damage_grade` from `building_damage`. Filter to `district_id = 4` (Gorkha). Add a `LIMIT 10` for initial inspection.


In [ ]:
district_id = 4

conn = duckdb.connect('./nepal.sqlite')

query = f"""
    SELECT
        h.*,
        s.*,
        i.vdcmun_id,
        d.damage_grade
    FROM household_demographics h
    JOIN id_map i ON h.household_id = i.household_id
    JOIN building_structure s ON i.building_id = s.building_id
    JOIN building_damage d ON i.building_id = d.building_id
    WHERE i.district_id = {district_id}
    LIMIT 5
"""

df_joined = conn.execute(query).df()
conn.close()

print(f"Shape: {df_joined.shape}")
print(f"\nColumns: {df_joined.columns.tolist()}")
print("\nFirst few rows:")
print(df_joined.head())

---

## Part 9: Loading the Full Dataset

The limited JOIN above confirms the query structure. Now we load the complete Gorkha dataset (no `LIMIT`), create the binary `severe_damage` target, and drop the leaky post-earthquake columns.

**What `wrangle` steps happen here:**

1. **Load all rows** for district 4 (no LIMIT)
2. **Create binary target**: `severe_damage = 1` if `damage_grade` in {'Grade 4', 'Grade 5'}, else 0
3. **Drop leaky columns**: `post_eq_floor_count`, `post_eq_roof_type`, `damage_grade` (raw string form), `building_id` (just an ID)
4. **Drop missing values**: `df.dropna()` ensures a complete feature matrix

> ⚠️ **We drop `damage_grade` after creating `severe_damage`.** The raw `damage_grade` string (e.g., "Grade 4") is essentially the target in another form — keeping it would be direct target leakage.

**Code Task 4.4.3.1**: Load the complete joined dataset for Gorkha (district_id = 4, no LIMIT). Create the binary `severe_damage` column. Drop post-earthquake columns and `damage_grade`. Drop rows with missing values. Store in `df`.


In [ ]:
conn = duckdb.connect('./nepal.sqlite')

query = """
    SELECT
        h.*,
        s.*,
        i.vdcmun_id,
        d.damage_grade
    FROM household_demographics h
    JOIN id_map i ON h.household_id = i.household_id
    JOIN building_structure s ON i.building_id = s.building_id
    JOIN building_damage d ON i.building_id = d.building_id
    WHERE i.district_id = 4
"""

# Load data
df = conn.execute(query).df()
conn.close()

# Create binary target, set index, and drop leaky columns
cols_to_drop = [col for col in df.columns if 'post_eq' in col] + \
               ['building_id', 'damage_grade', 'count_floors_pre_eq']

df = (df
  .assign(severe_damage=df['damage_grade']           # <--- Create binary target column
    .str.contains('Grade 4|Grade 5')                 # <--- Check if grade contains 4 or 5
    .astype(int))                                    # <--- Convert to 0/1 integer
  .set_index('household_id')                         # <--- Use household_id as row index
  .drop(columns=cols_to_drop))                       # <--- Remove post-earthquake and other leaky columns

print(f"Final shape: {df.shape}")
print(f"Severe damage rate: {df['severe_damage'].mean():.2%}")

---

## Part 10: Handling High-Cardinality Features — Caste

The `caste_household` column is a high-cardinality categorical feature with 100+ distinct values. Before we can use it in a model, we need to understand and reduce its cardinality.

**Why does high cardinality cause problems?**

| Problem | Explanation |
|---------|-------------|
| **Too many OHE columns** | 100+ castes → 100+ binary features, most with very few 1s |
| **Rare categories** | Castes with 5-10 occurrences give the model too little data to learn from |
| **Overfitting** | Rare categories learned from training data may not generalize |
| **Encoding failures** | Test data may contain caste values not seen during training |

**Our solution:** keep the top 10 most frequent castes, group the rest as "Other". This:
- Reduces 100+ categories to 11 (tractable cardinality)
- Preserves variation in the most common caste groups (where data is abundant)
- Handles rare categories gracefully without dropping rows

**Code Task 4.4.4.1**: Check the number of unique values in each categorical column. Which columns have high cardinality?


In [ ]:
# Check unique values in categorical columns
cat_cols = df.select_dtypes(include=['object', 'str']).columns       # <--- columns
cardinality = df[cat_cols].nunique().sort_values(ascending=False)    #<--- cat_cols

print("Categorical feature cardinality:")
print(cardinality)
print(f"\nTop 10 most common castes:")
print(df['caste_household'].value_counts().head(10))

Now we apply the top-10 caste grouping strategy.

> 💡 **Choosing the cutoff (top 10):** 10 is a reasonable starting point — it captures the main caste groups in Gorkha (Gurung, Magar, Tamang, Chhetri, Brahmin, and others) while keeping the feature space manageable. The exact cutoff is a hyperparameter you could tune: top 5 reduces cardinality further, top 20 preserves more signal from mid-frequency groups.

**Code Task 4.4.4.2**: Identify the top 10 most frequent castes. Create a new column where rare castes are replaced with `'Other'`. Verify that `caste_household` now has 11 unique values.


In [ ]:
# Get top 10 castes
top_10_castes = (df['caste_household']
  .value_counts()                   # <--- Count frequency of each caste
  .head(10)                         # <--- Keep only top 10 most common
  .index)                           # <--- Extract caste names

# Group rare castes as "Other"
df = (df
  .assign(caste_household=df['caste_household'].apply(
      lambda c: c if c in top_10_castes else 'Other'  # <--- Keep top 10 castes, group rest as "Other"
  )))

print(f"Unique castes after grouping: {df['caste_household'].nunique()}")
print("\nCaste distribution:")
print(df['caste_household'].value_counts())

✅ **You may now attempt Multiple Choice Question 4.4.4.1**

> 📊 **Interpreting the caste distribution:** the top castes in Gorkha include Gurung, Magar, and Tamang — all indigenous (Janajati) groups historically from this region. Chhetri and Brahmin (hill caste groups) are also present. The "Other" category aggregates smaller groups. This distribution will matter when we analyze damage rates by caste.

---

## Part 11: Preparing for Modeling

With the data cleaned and the high-cardinality caste column handled, we prepare the modeling inputs:

1. **Define target** (`severe_damage`) and **features** (all other columns)
2. **Split train/test** before any encoding
3. **Encode** inside a Pipeline (fitting only on training data)

**Which columns are features?** All columns except `severe_damage`. Note that `vdcmun_id` (municipality ID) is included — it's a legitimate pre-earthquake feature (geographic location) and may contain useful signal about local building practices and geology.

**Code Task 4.4.5.1**: Define `target = 'severe_damage'`. Create `features` as a list of all column names except the target. Create `X` (feature matrix) and `y` (target vector).


In [ ]:
# Define target and features
target = 'severe_damage'
features = [col for col in df.columns if col != target]

# Create X and y
X = df[features]   # <-- feature matrix
y = df[target]   # <-- target vector

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"\nFeatures include: {features[:5]}...")

Now split the data into train and test sets. We use `stratify=y` to ensure both sets have the same 64%/36% class distribution as the full dataset — this is especially important when classes are imbalanced.

> 📌 **`stratify=y` ensures representativeness.** Without stratification, a random split might put 70% of the severe cases in the training set and only 50% in the test set, creating an artificially easy test set. Stratification maintains the original class proportions in both splits.

**Code Task 4.4.5.2**: Split data into train (80%) and test (20%) sets using `random_state=42`. Print the shape of each split to verify.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training: {X_train.shape[0]} samples")
print(f"Test: {X_test.shape[0]} samples")

---

## Part 12: Building the Model

With the train/test split in place, we build two models:

1. **Baseline**: always predict the majority class ("severe") — establishes the floor
2. **Logistic Regression** with OHE: our model with both structural and demographic features

**Why `OneHotEncoder` (not `OrdinalEncoder`) for L4's LR?**

Logistic Regression computes `z = β₀ + β₁x₁ + β₂x₂ + ...` — a weighted sum. With ordinal integers (0, 1, 2 for foundation types), the model would treat category 2 as "twice as important" as category 1, which is incorrect for nominal categories like `caste_household`.

OHE creates one binary column per category, giving each its own independent coefficient.

> 💡 **`use_cat_names=True`** in `category_encoders.OneHotEncoder` preserves the original category names in the encoded column names (e.g., `caste_household_Gurung` instead of `caste_household_0`). This is essential for interpreting feature importances.

**Code 4.4.6.1**: Calculate the baseline accuracy (majority-class predictor) on the training set.


In [ ]:
baseline_acc = y_train.value_counts(normalize=True).max()
print(f"Baseline Accuracy: {baseline_acc:.4f}")

Now build the full Logistic Regression model with OneHotEncoder, using a Pipeline to prevent encoding leakage.

**Pipeline structure:**
```
OHE (fit on X_train only) → LogisticRegression
```

The Pipeline ensures:
1. `fit()` calls `OHE.fit_transform()` on training data, then `LR.fit()` on encoded training data
2. `predict()` calls `OHE.transform()` on new data (no re-fitting), then `LR.predict()`

This is the correct leakage-free workflow.

**Code 4.4.6.2**: Build a `Pipeline` with `OneHotEncoder(use_cat_names=True)` and `LogisticRegression(max_iter=1000)`. Fit on training data. Calculate training and test accuracy.


In [ ]:
# Build model
model = Pipeline([
    ('encoder', OneHotEncoder(use_cat_names=True)),
    ('classifier', LogisticRegression(max_iter=1000))
])

# Fit model
model.fit(X_train, y_train)

# Evaluate
train_acc = accuracy_score(y_train, model.predict(X_train))
test_acc = accuracy_score(y_test, model.predict(X_test))

print(f"Training Accuracy: {train_acc:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")
print(f"Baseline Accuracy: {baseline_acc:.4f}")

---

## Part 13: Feature Importance — What Drives Damage?

Logistic Regression coefficients tell us which features the model weighted most heavily. Converted to **odds ratios** (`exp(coefficient)`), they have a direct interpretation:

- **Odds ratio > 1**: feature is associated with higher odds of severe damage
- **Odds ratio < 1**: feature is associated with lower odds of severe damage (protective factor)
- **Odds ratio = 1**: feature has no effect on the odds

**Example interpretation:**
```
caste_Gurung coefficient = -0.35 → odds ratio = exp(-0.35) = 0.70
Interpretation: Gurung households have 30% lower odds of severe damage
compared to the baseline category, holding all other features constant.
```

> ⚠️ **"Holding all other features constant" is the key phrase.** A negative coefficient for a caste doesn't mean that caste is "inherently protected" — it means that, controlling for building structure and location, there's a residual difference. The mechanisms (building maintenance practices, self-built retrofitting) require domain expertise to interpret.

**Code 4.4.7.1**: Extract `coef_` from the fitted LogisticRegression. Compute odds ratios. Create a horizontal bar chart showing the top 10 features by absolute coefficient value.


In [ ]:
# Get feature names and coefficients
feature_names = model.named_steps['encoder'].get_feature_names_out()
coefficients = model.named_steps['classifier'].coef_[0]

# Create odds ratios
odds_ratios = pd.Series(np.exp(coefficients), index=feature_names).sort_values()

print("Top 10 features INCREASING odds of severe damage:")
print(odds_ratios.tail(10))

# Plot
fig, ax = plt.subplots(figsize=(9, 6))
odds_ratios.tail(10).plot(kind='barh', ax=ax)
ax.set_xlabel('Odds Ratio')
ax.set_title('Top 10 Risk Factors for Severe Damage')
ax.axvline(x=1, color='red', linestyle='--', label='No Effect')
ax.legend()
plt.tight_layout()
plt.show()

---

## Part 14: Municipal Analysis — Where Is Damage Highest?

Beyond individual building predictions, we want to understand **where** damage is concentrated. The `vdcmun_id` column identifies the municipality (Village Development Committee / Municipality) — the local administrative unit.

**Why municipal-level analysis matters:**

1. **Aid targeting:** post-earthquake relief is often distributed at the municipality level — knowing which municipalities had highest damage rates guides resource allocation
2. **Spatial confounding:** if demographic patterns (caste distribution) align with geographic damage patterns, some of the apparent caste-damage relationship may be geographic in origin
3. **Policy design:** building code enforcement and retrofitting programs operate at the local government level

**What to look for in the damage-by-municipality plot:**
- The range of damage rates (some municipalities: near 100% severe; others: near 30%)
- Whether high-damage municipalities are geographically clustered or dispersed
- Whether municipalities with high damage rates also have specific demographic compositions

**Code 4.4.8.1**: Create a DataFrame showing severe damage rate (mean of `severe_damage`) by municipality. Which municipalities have the highest and lowest damage rates?


In [ ]:
# Damage by municipality
damage_by_municipality = df.groupby('vdcmun_id')['severe_damage'].agg(['mean', 'count'])
damage_by_municipality.columns = ['damage_rate', 'count']
damage_by_municipality = damage_by_municipality.sort_values('damage_rate', ascending=False)

print("Top 10 municipalities by damage rate:")
print(damage_by_municipality.head(10))

# Plot
fig, ax = plt.subplots(figsize=(9, 6))
damage_by_municipality['damage_rate'].head(15).plot(kind='bar', ax=ax)
ax.set_xlabel('Municipality ID')
ax.set_ylabel('Severe Damage Rate')
ax.set_title('Severe Damage Rate by Municipality')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

### Protective Factors: The Smallest Coefficients

Features with the **most negative** coefficients are the strongest **protective factors** — they are associated with lower odds of severe damage. These tell us what characteristics protected buildings and households from the worst outcomes.

> 📌 **Protective factors are as important as risk factors for policy.** If RC foundations strongly predict "not severe," this tells policymakers that replacing mud-mortar foundations with RC in future construction will reduce damage in the next earthquake.

**Code 4.4.8.2**: Create a horizontal bar chart showing the **10 features with the smallest coefficients** (most protective). These are the features that most strongly decrease the odds of severe damage.


In [ ]:
print("Top 10 features DECREASING odds of severe damage (protective factors):")
print(odds_ratios.head(10))

# Plot
fig, ax = plt.subplots(figsize=(9, 6))
odds_ratios.head(10).plot(kind='barh', ax=ax, color='green')
ax.set_xlabel('Odds Ratio')
ax.set_title('Top 10 Protective Factors (Lowest Odds Ratios)')
ax.axvline(x=1, color='red', linestyle='--', label='No Effect')
ax.legend()
plt.tight_layout()
plt.show()

### Spatial Pattern: Damage Rate by Municipality ID

Plotting damage rate by municipality ID (sorted by ID, not by damage rate) reveals the **spatial pattern** — if geographically adjacent municipalities have similar damage rates, this suggests geological clustering.

> 🔍 **What the line plot reveals:** if the plot shows smooth spatial transitions (neighboring municipalities have similar rates), this suggests the primary driver is local geology (amplification of seismic waves in certain valleys) rather than random variation. If the plot is jagged, damage is driven by building-level factors rather than geography.

**Code 4.4.8.3**: Create a line plot of `damage_by_municipality` sorted by municipality ID. X-axis: municipality ID; Y-axis: severe damage rate. This reveals the spatial pattern.


In [ ]:
# Sort by municipality ID for line plot
damage_by_municipality_sorted = damage_by_municipality.sort_index()

# Create line plot
fig, ax = plt.subplots(figsize=(9, 6))
ax.plot(damage_by_municipality_sorted.index,
        damage_by_municipality_sorted['damage_rate'],
        marker='o', linewidth=2, markersize=8)
ax.set_xlabel('Municipality ID')
ax.set_ylabel('% of Total Households with Severe Damage')
ax.set_title('Household Damage by Municipality')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---

## Part 15: Demographics by Municipality — Caste Analysis

Now we connect the spatial damage analysis to demographic composition. If municipalities with high damage rates also have concentrated populations of specific caste groups, this reveals which communities bore the greatest burden of the earthquake.

**The analytical question:**

```
Hypothesis: Municipalities with high Gurung concentrations
            also have higher damage rates
→ If true: Gurung households are disproportionately in high-damage areas
→ Mechanism: Geography? Building quality? Historical access to construction resources?
```

> 📌 **This is an observational analysis, not a causal claim.** We are quantifying correlations between demographic composition and damage rates — not establishing that caste *causes* damage. The causal pathways are economic and structural, not biological or cultural.

**Code 4.4.9.1**: For each municipality, calculate the **proportion of Gurung households** (`df['caste_household'] == 'Gurung'`). Join this to `damage_by_municipality`. Plot: x-axis = Gurung proportion, y-axis = severe damage rate. Is there a correlation?


In [ ]:
# Calculate proportion of Gurung households by municipality
gurung_by_municipality = (
    df[df['caste_household'] == 'Gurung']
    .groupby('vdcmun_id')
    .size()
)

# Total households by municipality
total_by_municipality = df.groupby('vdcmun_id').size()

# Calculate proportion
damage_by_municipality['gurung_pct'] = (
    gurung_by_municipality / total_by_municipality
).fillna(0)

print("Gurung population by municipality:")
print(damage_by_municipality[['damage_rate', 'gurung_pct']].sort_values('gurung_pct', ascending=False))

### Kumal Households: A Second Caste Analysis

We repeat the analysis for **Kumal** households — a caste group with different historical geographic and economic circumstances. Comparing two caste groups helps distinguish patterns that are:

- **Caste-specific**: one group has a pattern the other doesn't → likely due to that group's specific geography or building practices
- **Universal**: both groups show similar patterns → likely driven by a common factor (geography, building age, etc.)

> 💡 **Why compare multiple groups?** A single caste-damage correlation could be coincidental. Seeing consistent patterns across multiple caste groups with different geographic and socioeconomic profiles builds a more robust picture of the equity implications.

**Code 4.4.9.2**: Calculate the proportion of **Kumal** households in each municipality. Replace NaN with 0 (for municipalities with no Kumal households). Add as a column to `damage_by_municipality`. Plot: x-axis = Kumal proportion, y-axis = severe damage rate.


In [ ]:
# Calculate proportion of Kumal households by municipality
kumal_by_municipality = (
    df[df['caste_household'] == 'Kumal']
    .groupby('vdcmun_id')
    .size()
)

# Calculate proportion (NaN will become 0)
damage_by_municipality['kumal_pct'] = (
    kumal_by_municipality / total_by_municipality
).fillna(0)

print("Kumal population by municipality:")
print(damage_by_municipality[['damage_rate', 'kumal_pct']].sort_values('kumal_pct', ascending=False))

# Show correlation
print(f"\nCorrelation between damage rate and Gurung population: {damage_by_municipality['damage_rate'].corr(damage_by_municipality['gurung_pct']):.3f}")
print(f"Correlation between damage rate and Kumal population: {damage_by_municipality['damage_rate'].corr(damage_by_municipality['kumal_pct']):.3f}")

✅ **You may now attempt Multiple Choice Question 4.4.8.1**

> 📊 **Interpreting the caste-damage scatter plots:** do high-damage municipalities have systematically higher or lower proportions of specific caste groups? A positive correlation suggests that group lives disproportionately in high-damage areas. The strength of correlation tells us how much of the spatial damage pattern is explained by demographic composition.

---

## Summary and Discussion

This lesson combined SQL skills, exploratory data analysis, and classification modeling to investigate a fundamental equity question: **who gets hit hardest by the earthquake, and why?**

### What You Built and Learned

| Concept | Key Takeaway |
|---------|-------------|
| **ML framing** | Target definition, binary classification, class imbalance as baseline problem |
| **Target leakage** | Post-earthquake features and target proxies must be excluded; encoder must fit only on training data |
| **SQL JOINs** | Four-table INNER JOIN connects demographic, structural, and damage data |
| **HAVING** | Filters groups after aggregation; `WHERE` filters rows before grouping |
| **Table aliases** | `h`, `s`, `i`, `d` shorthand prevents long table name repetition |
| **High-cardinality handling** | Top-10 + "Other" reduces 100+ caste values to tractable feature space |
| **Stratified split** | Maintains class proportions in both train and test sets |
| **OneHotEncoder** | Required for logistic regression on nominal categorical features |
| **Odds ratios** | exp(coefficient) — interpretable as multiplicative change in damage odds |
| **Municipal analysis** | Spatial heterogeneity in damage rates; some municipalities 3× higher than others |
| **Caste analysis** | Demographic composition correlates with municipal damage rates |

### Key Findings

- **Model performance:** adding demographic features to structural features achieves similar accuracy to structure-only models (~71-72%). Demographic features do not substantially improve predictive accuracy, which suggests structural features capture most of the predictable variation.
- **Top risk factors:** `foundation_type`, `ground_floor_type`, and `age_building` dominate the coefficient magnitude — consistent with Lessons 2 and 3. Demographic features have smaller (but non-zero) coefficients.
- **Municipality variation:** severe damage rates vary significantly across municipalities. Some municipalities show rates near 90%; others near 30%. This spatial heterogeneity likely reflects a combination of local geology and building stock composition.
- **Demographic correlation:** certain caste groups are concentrated in high-damage municipalities. This correlation is real in the data, but its mechanism is structural and geographic — not attributable to caste identity itself.

### Equity Discussion

**The core questions this analysis raises:**

1. **Does the model show that certain demographic groups are more affected by the earthquake?** Yes — at the municipal level, caste composition correlates with damage rates. This reflects that lower-caste communities have historically been concentrated in areas with older, weaker building stock.

2. **Or does it reflect that certain groups live in buildings with poorer structural features?** Largely yes — when structural features are controlled for, the residual caste effect is smaller. The primary driver of damage is building quality, and building quality correlates with caste through historical economic exclusion.

3. **What are the policy implications?**
   - Reconstruction programs should prioritize the most damaged communities by building type AND by geographic location — not by caste directly
   - But because caste and building quality correlate, programs targeting old mud-mortar buildings in high-damage municipalities will disproportionately reach marginalized communities
   - Anti-discrimination in reconstruction aid is essential: lower-caste households should not receive lower-quality replacement buildings or slower reconstruction

**The Data Science Ethics Framework:**

| Principle | Application to Nepal Earthquake |
|-----------|-------------------------------|
| **Fairness** | Are our damage predictions equally accurate for all caste groups? Disparate accuracy across groups creates disparate aid |
| **Transparency** | Can reconstruction authorities understand why specific buildings are flagged? Tree visualizations (Lesson 3) support this |
| **Accountability** | Who is responsible when the model misclassifies a severely damaged building as safe? This must be defined before deployment |
| **Non-discrimination** | Demographic features should not be used as direct inputs to allocation decisions — but understanding demographic patterns helps identify systemic gaps |

> 📌 **The purpose of this analysis is to make inequalities visible, not to perpetuate them.** Data science cannot eliminate the structural inequalities that shaped the damage patterns. But it can ensure that reconstruction efforts are allocated where the need is greatest — and that aid programs understand which communities are at highest risk.

### Discussion Questions

1. In the municipal analysis, some municipalities show near-90% severe damage rates. What factors might explain this extreme concentration of damage in specific locations?
2. The model's accuracy does not improve much when demographic features are added to structural features. What does this suggest about the relative importance of structural vs demographic factors in explaining damage?
3. If we used `caste_household` as a feature in a reconstruction aid allocation model (not a damage prediction model), what ethical concerns would arise?
4. Why is `HAVING COUNT(*) > 100` used when analyzing caste distribution, rather than `WHERE COUNT(*) > 100`?
5. How would the analysis change if we used `vdcmun_id` as a categorical (OHE'd) feature instead of a numerical feature?
6. The odds ratio for `foundation_type_RC` is much less than 1 (protective). Does this mean RC foundations always protect buildings from severe damage? What factors might reduce its protective effect?

### Next Steps: Lesson 5 (End-to-End Assignment)

In Lesson 5, you will complete an end-to-end assignment applying everything from Lessons 1-4:

- Connect to the Nepal database with DuckDB (Lesson 1 SQL skills)
- Build a complete data pipeline with wrangling, splitting, and encoding (Lessons 2-4)
- Train and compare Logistic Regression and Decision Tree models (Lessons 2-3)
- Evaluate with precision, recall, and ROC/AUC (Lesson 2)
- Make and justify your final model choice with ethical considerations (Lesson 4)

> ➡️ Lesson 5 is the culmination of Project 4 — your opportunity to demonstrate mastery of the complete classification workflow from database to deployed model decision.
